In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StringType

In [0]:
df_taxas_silver = spark.table("projeto_seguranca_rj.bronze.taxas_municipio")

In [0]:
df_taxas_silver.printSchema()

In [0]:
df_taxas_silver = (
    df_taxas_silver
    .withColumn(
        "fmun_cod",
        F.col("fmun_cod").cast("string")
    )
)

In [0]:
colunas_taxas = [
    "hom_doloso",
    "lesao_corp_morte",
    "latrocinio",
    "cvli",
    "hom_por_interv_policial",
    "letalidade_violenta",
    "tentat_hom",
    "lesao_corp_dolosa",
    "estupro",
    "hom_culposo",
    "lesao_corp_culposa",
    "roubo_transeunte",
    "roubo_celular",
    "roubo_em_coletivo",
    "roubo_rua",
    "roubo_veiculo",
    "roubo_carga",
    "roubo_comercio",
    "roubo_residencia",
    "roubo_banco",
    "roubo_cx_eletronico",
    "roubo_conducao_saque",
    "roubo_apos_saque",
    "roubo_bicicleta",
    "outros_roubos",
    "total_roubos",
    "furto_veiculos",
    "furto_transeunte",
    "furto_coletivo",
    "furto_celular",
    "furto_bicicleta",
    "outros_furtos",
    "total_furtos",
    "sequestro",
    "extorsao",
    "sequestro_relampago",
    "estelionato",
    "apreensao_drogas",
    "posse_drogas",
    "trafico_drogas",
    "apreensao_drogas_sem_autor",
    "recuperacao_veiculos",
    "apf",
    "aaapai",
    "cmp",
    "cmba",
    "ameaca",
    "pessoas_desaparecidas",
    "encontro_cadaver",
    "encontro_ossada",
    "pol_militares_mortos_serv",
    "pol_civis_mortos_serv",
    "registro_ocorrencias"
]

for coluna in colunas_taxas:
    df_taxas_silver = (
        df_taxas_silver
        .withColumn(
            coluna,
            F.regexp_replace(
                F.col(coluna),
                ",",
                "."
            )
        )
    )

for coluna in colunas_taxas:
    df_taxas_silver = (
        df_taxas_silver
        .withColumn(
            coluna,
            F.col(coluna).cast("double")
        )
    )

In [0]:
df_taxas_silver.printSchema()

In [0]:
df_taxas_silver.select(
    [
        F.count(
            F.when(
                F.col(c).isNull(),
                c
            )
        ).alias(c)
        for c in df_taxas_silver.columns
    ]
).show()

In [0]:
df_taxas_silver = (
    df_taxas_silver
    .withColumn(
        "regiao",
        F.coalesce(
            F.col("regiao"),
            F.lit("NAO_INFORMADO")
        )
    )
)

In [0]:
df_taxas_silver.select(
    F.count(
        F.when(
            F.col("regiao").isNull(),
            "regiao"
        )
    ).alias("nulos_regiao")
).show()

In [0]:
df_taxas_silver.groupBy("regiao").count().show()

In [0]:
df_taxas_silver.groupBy(
    "fmun_cod",
    "ano",
    "mes"
).count() \
.filter(
    F.col("count") > 1
).show()

In [0]:
df_taxas_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "projeto_seguranca_rj.silver.taxas_municipio"
    )